 ## Setting Up Your Azure AI Key and Endpoint

In [ ]:
%pip install azure-ai-textanalytics


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [12]:
from azure.core.credentials import AzureKeyCredential
from azure.ai.textanalytics import TextAnalyticsClient

In [ ]:
key = "<REDACTED_API_KEY>"
endpoint = "https://temp-m2web-ai-search-pr-resource.services.ai.azure.com/api/projects/temp-m2web-ai-search-project"

# 1. FORCE CLEAN THE STRINGS (Ensures no hidden spaces or tabs)
clean_key = key.strip()
clean_endpoint = endpoint.strip().split(".com")[0] + ".com"
print(f"Connecting to: {clean_endpoint}...")

Connecting to: https://temp-m2web-ai-search-pr-resource.services.ai.azure.com...


In [15]:
# 2. Re-initialize with the explicit version your SDK liked (2023-04-01)
client = TextAnalyticsClient(
    endpoint=clean_endpoint, 
    credential=AzureKeyCredential(clean_key),
    api_version="2023-04-01" # This matches your SDK's highest supported version
)

## Loading Data for Text Analysis

In [16]:
import os

lyrics_folder = "rush_lyrics"
lyrics = []

In [17]:
for filename in os.listdir(lyrics_folder):
    if filename.endswith(".txt"):
        with open(os.path.join(lyrics_folder, filename), "r") as file:
            lyrics.append({"id": filename, "text": file.read()})

In [18]:
# Confirm the data is loaded correctly
print(f"Loaded {len(lyrics)} songs.")

Loaded 10 songs.


## Extractive Summarization

In [19]:
target_lyrics = [song['text'] for song in lyrics[:3]]

print(f"Generating extractive summaries for {len(target_lyrics)} songs...")

poller = client.begin_extract_summary(target_lyrics)
results = poller.result()

for song_metadata, result in zip(lyrics[:3], results):
    print(f"--- {song_metadata['id']} ---")
    if not result.is_error:
        for sentence in result.sentences:
            print(f"- {sentence.text}")
    else:
        print(f"Error: {result.error.code} - {result.error.message}")
    print("-" * 30)

Generating extractive summaries for 3 songs...
--- a_passage_to_bangkok.txt ---
- Title: A Passage to Bangkok
- We're on the train to Bangkok
- Aboard the Thailand Express
------------------------------
--- by_tor_and_the_snow_dog.txt ---
- Title: By-Tor and the Snow Dog
- Prince By-Tor takes the cavern to the north light
- By-Tor and the Snow Dog
------------------------------
--- different_strings.txt ---
- Album: Permanent Waves
- To be found within a song
- Along with our naivete
------------------------------


## Abstractive Summarization

In [20]:
print(f"Generating abstractive summaries...")

poller = client.begin_abstract_summary(target_lyrics)
results = poller.result()

for song_metadata, result in zip(lyrics[:3], results):
    print(f"--- {song_metadata['id']} ---")
    if not result.is_error:
        for summary in result.summaries:
            print(f"Summary: {summary.text}")
    else:
        print(f"Error: {result.error.code} - {result.error.message}")
    print("-" * 30)

Generating abstractive summaries...
--- a_passage_to_bangkok.txt ---
Summary: The song "A Passage to Bangkok" from the album "2112" describes a vibrant journey through various countries before reaching the final destination in Bangkok. The travelers start in Bogota, then experience the allure of Jamaican dreams and Acapulco nights. They continue through Morocco and the East, arriving with the morning light. The journey includes stays in Lebanon, where they stay up late, and Nepal, with its distinctive scent marking their arrival. The song's chorus emphasizes their commitment to only stopping for the most enjoyable experiences, highlighted by their travel on the Thailand Express. Throughout, the song paints a picture of cultural richness and adventure, culminating in the sensory experiences of their final destinations.
------------------------------
--- by_tor_and_the_snow_dog.txt ---
Error: InvalidDocument - Appropriate summarization cannot be generated.
------------------------------


## Customizing Summarization Options - Sentence Count Limits

In [21]:
poller = client.begin_extract_summary(target_lyrics, max_sentence_count=2)
results = poller.result()

for song_metadata, result in zip(lyrics[:3], results):
    print(f"--- {song_metadata['id']} (Max 2 Sentences) ---")
    if not result.is_error:
        for sentence in result.sentences:
            print(f"- {sentence.text}")
    else:
        print(f"Error: {result.error.code} - {result.error.message}")
    print("-" * 30)

--- a_passage_to_bangkok.txt (Max 2 Sentences) ---
- Title: A Passage to Bangkok
- Aboard the Thailand Express
------------------------------
--- by_tor_and_the_snow_dog.txt (Max 2 Sentences) ---
- Title: By-Tor and the Snow Dog
- By-Tor and the Snow Dog
------------------------------
--- different_strings.txt (Max 2 Sentences) ---
- Album: Permanent Waves
- Along with our naivete
------------------------------
